In [1]:
from pydantic import BaseModel

# 1. Define the parameters for a single generator
class Generator(BaseModel):
    name: str
    cost_per_mw: float
    max_capacity_mw: float

# 2. Define the overall problem structure
class DispatchProblem(BaseModel):
    objective_type: str  # e.g., "minimize cost"
    total_demand_mw: float
    generators: list[Generator]

# 3. Create a dummy object to test if Pydantic is working
dummy_data = DispatchProblem(
    objective_type="minimize cost",
    total_demand_mw=150.0,
    generators=[
        Generator(name="Gen_1", cost_per_mw=20.0, max_capacity_mw=100.0),
        Generator(name="Gen_2", cost_per_mw=30.0, max_capacity_mw=120.0)
    ]
)

print(f"Environment Success! Schema defined. Grid demands {dummy_data.total_demand_mw} MW.")

Environment Success! Schema defined. Grid demands 150.0 MW.


In [6]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load the hidden API key from your .env file
load_dotenv()

# 2. Initialize the Gemini client 
client = genai.Client()

# 3. Define the plain-text problem (Unstructured Data)
prompt_text = """
The grid operator needs to meet a total demand of 150 MW. 
There are two generators available. Generator 1 costs $20 per MW 
to run and has a maximum capacity of 100 MW. Generator 2 costs $30 
per MW to run and has a maximum capacity of 120 MW. 
The objective is to minimize the total generation cost.
"""

# 4. Ask Gemini to read the text and fill out the Pydantic form
print("Sending text to Gemini...")
response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=prompt_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DispatchProblem,
        temperature=0.0, # We set this to 0 because we want exact facts, not creativity
    ),
)

# 5. Convert the JSON response back into our Pydantic object
extracted_data = DispatchProblem.model_validate_json(response.text)

print("\n--- Extraction Success! ---")
print(f"Objective: {extracted_data.objective_type}")
print(f"Demand: {extracted_data.total_demand_mw} MW")
for gen in extracted_data.generators:
    print(f"Found {gen.name}: Max {gen.max_capacity_mw}MW, Cost ${gen.cost_per_mw}/MW")



Sending text to Gemini...

--- Extraction Success! ---
Objective: minimize_cost
Demand: 150.0 MW
Found Generator 1: Max 100.0MW, Cost $20.0/MW
Found Generator 2: Max 120.0MW, Cost $30.0/MW


In [8]:
print(extracted_data)

objective_type='minimize_cost' total_demand_mw=150.0 generators=[Generator(name='Generator 1', cost_per_mw=20.0, max_capacity_mw=100.0), Generator(name='Generator 2', cost_per_mw=30.0, max_capacity_mw=120.0)]


In [17]:
import pyomo.environ as pyo
model = pyo.ConcreteModel()
model.total_demand_mw = pyo.Param(initialize = extracted_data.total_demand_mw)
model.cost_per_mw_1 = pyo.Param(initialize = extracted_data.generators[0].cost_per_mw)
model.cost_per_mw_2 = pyo.Param(initialize = extracted_data.generators[1].cost_per_mw)

model.g1 = pyo.Var(bounds=(0, extracted_data.generators[0].max_capacity_mw))
model.g2 = pyo.Var(bounds=(0, extracted_data.generators[1].max_capacity_mw))

model.c1 = pyo.Constraint(expr = model.g1 + model.g2 == model.total_demand_mw)
model.c2 = pyo.Constraint(expr = model.g1 + model.g2 == model.total_demand_mw)

model.f1 = pyo.Objective(expr = model.cost_per_mw_1 * model.g1 + model.cost_per_mw_2 * model.g2, sense=pyo.minimize)

solver = pyo.SolverFactory('glpk')
results = solver.solve(model)
print(f"generator 1: {pyo.value(model.g1)}")
print(f"generator 2: {pyo.value(model.g2)}")


generator 1: 100.0
generator 2: 50.0
